# Laboratorio de métodos de búsqueda

|                |   |
:----------------|---|
| **Nombre**     | Carlos Enrique Nieves Ochoa  |
| **Fecha**      | 28 de octubre de 2026 |
| **Expediente** | 758210  |

Modelos como regresión logística, máquinas de vectores de soporte, y preceptrones multicapa tienen una serie de argumentos que necesitan valores dados antes de poder usarse para ajustarse a un dataset. Estos argumentos se llaman hiperparámetros.

**¿Cuál es la diferencia entre un parámetro y un hiperparámetro?**

Un parámetro lo aprende el modelo a partir de los datos cuando hago `fit`. En la regresión logística son los coeficientes $\beta$ (`intercept_` y `coef_`) y en un perceptrón son los pesos y sesgos de cada neurona; yo no los elijo, salen de minimizar la función de pérdida, mientras que un hiperparámetro lo fijo yo antes de entrenar y el modelo no lo cambia durante el ajuste. Por ejemplo `C` en la regresión logística y en la SVM, el `kernel` y `gamma` de la SVM, o `hidden_layer_sizes`, `activation` y `alpha` del `MLPClassifier`. Como no se aprenden con los datos de entrenamiento, la única forma de escogerlos es probar varios valores y comparar cómo le va a cada modelo con datos que no vio, por ejemplo con validación cruzada.

Este laboratorio se centra en métodos de búsqueda para encontrar los mejores hiperparámetros para un modelo dado. Imagina que estás jugando batalla naval:

<center><img src="https://cdn11.bigcommerce.com/s-nehqfptb/images/stencil/670x993/products/40096/100569/battleship-game__73281.1768238533.jpg?c=2"></center>

Una forma de jugar podría ser ir de cuadro en cuadro hasta probar todo el tablero. El problema aquí es que se juega a contrarreloj, debido a que tu oponente puede hundir tus barcos antes de que pruebes todo el tablero.

Si no se tiene la preocupación/limitante de tiempo, probar todas las combinaciones es una opción:

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

**Entra a la documentación de `GridSearchCV` de `sklearn` y describe cómo funciona este objeto (entradas, salidas, parámetros).**

- **Qué hace:** `GridSearchCV` prueba todas las combinaciones de una rejilla de hiperparámetros. A cada combinación la califica con validación cruzada: ajusta el modelo con $k-1$ grupos, lo evalúa en el grupo que quedó fuera, repite con cada grupo y promedia. Al final se queda con la combinación de mejor promedio. Si la rejilla tiene $m$ combinaciones y uso $k$ grupos, entrena $m \times k$ modelos (más uno final).
- **Entradas (parámetros principales):**
  - `estimator`: el modelo o el pipeline que se va a ajustar.
  - `param_grid`: diccionario con el nombre de cada hiperparámetro y la lista de valores a probar (o una lista de diccionarios si hay rejillas distintas). Dentro de un pipeline el nombre lleva el paso, como `svc__C` o `mlpclassifier__hidden_layer_sizes`.
  - `scoring`: la métrica con la que se califica, por ejemplo `"roc_auc"`. Si no se da, usa el `score` del modelo (accuracy en clasificación).
  - `cv`: cuántos grupos usar (5 por defecto) o un objeto como `KFold(...)`. Con un número entero y un clasificador usa grupos estratificados.
  - `n_jobs`: cuántos núcleos usar en paralelo; `-1` usa todos.
  - `refit`: si es `True` (por defecto) vuelve a entrenar la mejor combinación con todos los datos de `fit`.
  - Otros: `return_train_score` para guardar también la calificación en entrenamiento, `verbose` para ver el avance y `error_score` para qué hacer si un ajuste falla.
- **Uso y salidas:** se ejecuta con `grid.fit(X, y)` y deja estos atributos:
  - `best_params_`: la mejor combinación.
  - `best_score_`: su calificación promedio en la validación cruzada.
  - `best_estimator_`: el modelo con la mejor combinación ya reentrenado (si `refit=True`).
  - `cv_results_`: diccionario con los resultados de todas las combinaciones (`params`, `mean_test_score`, `std_test_score`, `rank_test_score`, tiempos de ajuste, etc.), que se puede pasar a `pd.DataFrame`.
  - Además, `grid.predict()`, `grid.predict_proba()` y `grid.score()` usan directamente al mejor modelo.
- **Ventaja y costo:** si la mejor combinación está en la rejilla, la encuentra seguro; el problema es que el número de combinaciones se multiplica con cada hiperparámetro que agrego, así que se vuelve muy lento.

Otra forma de "buscar" podría ser tirar al azar y ver qué pasa. Esto sería otra forma de utilizar la fuerza bruta para encontrar información. 

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html

**Entra a la documentación de `RandomizedSearchCV` de `sklearn` y describe cómo funciona este objeto (entradas, salidas, parámetros).**

- **Qué hace:** `RandomizedSearchCV` funciona igual que `GridSearchCV` (valida cada combinación con validación cruzada y se queda con la mejor), pero en lugar de probar todas, **sortea** un número fijo de combinaciones. El costo ya no depende del tamaño de la rejilla sino de cuántas combinaciones le pida: entrena `n_iter` $\times k$ modelos.
- **Entradas (parámetros principales):**
  - `estimator`: el modelo o el pipeline.
  - `param_distributions`: diccionario con cada hiperparámetro y de dónde sortearlo. Puede ser una lista (cada valor tiene la misma probabilidad) o una distribución de `scipy.stats` que tenga método `rvs`, como `randint` o `loguniform`. Si todo son listas, sortea sin reemplazo; si hay alguna distribución, con reemplazo.
  - `n_iter`: cuántas combinaciones probar (10 por defecto). Es el que controla el balance entre tiempo y calidad de la búsqueda.
  - `scoring`, `cv`, `n_jobs`, `refit` y `verbose`: igual que en `GridSearchCV`.
  - `random_state`: semilla del sorteo, para que se repitan las mismas combinaciones cada vez que lo corro.
- **Salidas:** las mismas que `GridSearchCV`: `best_params_`, `best_score_`, `best_estimator_`, `cv_results_` (solo con las combinaciones sorteadas), y los métodos `predict`, `predict_proba` y `score`.
- **Ventaja y costo:** con pocas pruebas suele llegar a combinaciones buenas, sobre todo cuando muchas combinaciones dan resultados parecidos, pero no garantiza encontrar la mejor. Como dice la nota de abajo, cada sorteo es independiente: no aprende de lo que ya probó.

**Nota:** RandomizedSearchCV prueba puramente al azar. Si encontrara en uno de sus tiros que se atinó a un barco, no cambia el algoritmo de búsqueda para seguir buscando en esa zona. Para un proceso de búsqueda más refinado necesitaríamos un algoritmo que cambie su forma de decidir a partir de información anterior.

**Revisa la documentación de Optuna (https://optuna.org/). Instala la librería y escribe una guía de cómo se debe utlizar para un dataset dado.**

Optuna es una librería para buscar hiperparámetros que sí usa la información de las pruebas anteriores. Por defecto usa el muestreador **TPE** (*Tree-structured Parzen Estimator*), que es un tipo de optimización bayesiana: separa las pruebas que ya hizo en "buenas" y "malas", estima qué valores son más probables en cada grupo y propone el siguiente intento donde es más probable mejorar, con algo de exploración para no quedarse en un solo lugar. Es la misma idea del notebook de Optimización Bayesiana, donde pedíamos el siguiente punto según lo que el proceso gaussiano ya sabía. En la batalla naval sería seguir tirando cerca del barco que ya tocamos.

**Guía para usarlo con un dataset:**

1. **Preparar los datos** como siempre: cargar, limpiar, separar `X` y `y`, y armar el preprocesamiento (por ejemplo un `ColumnTransformer` dentro de un `Pipeline` para que el escalamiento se ajuste solo con los datos de entrenamiento de cada grupo).
2. **Escribir la función objetivo.** Es una función que recibe un `trial` (un intento), le pide a Optuna los valores de los hiperparámetros, arma el modelo, lo evalúa con validación cruzada y regresa la calificación. Los valores se piden con:
   - `trial.suggest_int("nombre", minimo, maximo)` para enteros (neuronas, número de capas),
   - `trial.suggest_float("nombre", minimo, maximo, log=True)` para reales como `alpha` o `learning_rate_init`,
   - `trial.suggest_categorical("nombre", [opciones])` para opciones como `activation` o `solver`.

   A diferencia de una rejilla, un hiperparámetro puede depender de otro: por ejemplo, solo pido las neuronas de la capa 3 si el número de capas es al menos 3.
3. **Crear el estudio** con `optuna.create_study(direction="maximize")` si la métrica es como el AUC (o `"minimize"` si es un error como el MSE). Para que se pueda repetir, se le da una semilla al muestreador: `sampler=optuna.samplers.TPESampler(seed=42)`.
4. **Optimizar** con `study.optimize(objective, n_trials=100)`, donde `n_trials` es cuántos intentos hacer (también se puede limitar por tiempo con `timeout` en segundos).
5. **Leer los resultados:** `study.best_params` (mejores hiperparámetros), `study.best_value` (su calificación), `study.best_trial` y `study.trials_dataframe()` con todos los intentos.
6. **Entrenar el modelo final** con `study.best_params` y, si se separó un conjunto de prueba, evaluarlo ahí.

Esqueleto general:

```python
import optuna

def objective(trial):
    c1 = trial.suggest_int("c1", 5, 10)
    alpha = trial.suggest_float("alpha", 1e-5, 1e-1, log=True)
    modelo = MLPClassifier(hidden_layer_sizes=(c1,), alpha=alpha)
    # ... validación cruzada y calcular la métrica ...
    return auc_promedio

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=100)
print(study.best_params, study.best_value)
```

Opcionalmente, Optuna puede cortar intentos que van mal antes de terminarlos (*pruning*) con `trial.report()` y `trial.should_prune()`, y por defecto imprime una línea por intento, lo que se puede apagar con `optuna.logging.set_verbosity(optuna.logging.WARNING)`.

### **Ejercicio**

Utiliza el dataset `College.csv` para intentar predecir si una universidad es privada a partir del resto de los factores.

Utiliza un perceptrón multicapa con hiperparámetros:
- Número de capas intermedias --> {2, 3, 4}
- Número de neuronas en capa 1 --> [5, 10]
- Número de neuronas en capa 2 --> [10, 20]
- Número de neuronas en capa 3 (si existe) --> [5, 20]
- Número de neuronas en capa 4 (si existe) --> [5, 10]

Utiliza k-folds cross-validation con $k=10$; optimiza el ROC-AUC de test.

Compara los 3 diferentes métodos de búsqueda descritos en este laboratorio.

**1. Librerías**

In [1]:
import numpy as np
import pandas as pd
import optuna
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import KFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import roc_auc_score

**2. Cargar y revisar los datos**

La primera columna del archivo trae el nombre de cada universidad sin encabezado, así que la quito porque no es un factor.

In [2]:
df = pd.read_csv("College.csv")
df = df.drop(columns=["Unnamed: 0"])
print(df.shape)
df.head()

(777, 18)


,Private,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate
0,Yes,1660,1232,721,23,52,2885,537,7440,3300,450,2200,70,78,18.1,12,7041,60
1,Yes,2186,1924,512,16,29,2683,1227,12280,6450,750,1500,29,30,12.2,16,10527,56
2,Yes,1428,1097,336,22,50,1036,99,11250,3750,400,1165,53,66,12.9,30,8735,54
3,Yes,417,349,137,60,89,510,63,12960,5450,450,875,92,97,7.7,37,19016,59
4,Yes,193,146,55,16,44,249,869,7560,4120,800,1500,76,72,11.9,2,10922,15


In [3]:
# ¿Falta algo, hay filas repetidas y cuántas son privadas?
print(df.isna().sum().sum())
print("Duplicados:", df.duplicated().sum())
print(df["Private"].value_counts())

0
Duplicados: 0
Private
Yes    565
No     212
Name: count, dtype: int64


**3. Separar los datos y armar el modelo**

Convierto `Private` a 1 (privada) y 0 (pública). Todos los factores son numéricos y están en escalas muy distintas (`Apps` llega a miles y `S.F.Ratio` es de decenas), así que los escalo dentro del pipeline para que el escalamiento se ajuste solo con los grupos de entrenamiento de cada vuelta del K-Fold.

In [4]:
X = df.drop(columns=["Private"])
y = (df["Private"] == "Yes").astype(int)

numerical_cols = list(X.columns)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols)
    ],
    remainder="drop"
)

mlp = MLPClassifier(max_iter=500, random_state=42)
model = make_pipeline(preprocessor, mlp)

# K-Fold con k = 10; el mismo para los tres métodos
kf = KFold(n_splits=10, shuffle=True, random_state=42)

Uso el mismo `kf` en los tres métodos para que todas las redes se califiquen con los mismos 10 grupos y la comparación sea justa. `max_iter=500` y `random_state=42`.

**4. Armar la rejilla**

Con ciclos armo todas las redes que pide el ejercicio: con 2 capas ($6 \times 11 = 66$), con 3 capas ($6 \times 11 \times 16 = 1056$) y con 4 capas ($6 \times 11 \times 16 \times 6 = 6336$).

In [5]:
capas = []

# 2 capas intermedias
for c1 in range(5, 11):
    for c2 in range(10, 21):
        capas.append((c1, c2))

# 3 capas intermedias
for c1 in range(5, 11):
    for c2 in range(10, 21):
        for c3 in range(5, 21):
            capas.append((c1, c2, c3))

# 4 capas intermedias
for c1 in range(5, 11):
    for c2 in range(10, 21):
        for c3 in range(5, 21):
            for c4 in range(5, 11):
                capas.append((c1, c2, c3, c4))

print(len(capas))
print(capas[:3], capas[-1])

param_grid = {"mlpclassifier__hidden_layer_sizes": capas}

7458
[(5, 10), (5, 11), (5, 12)] (10, 20, 20, 10)


**5. Método 1: `GridSearchCV`**

Pruebo las 7,458 redes, cada una con el K-Fold de 10, así que son 74,580 entrenamientos. Por eso uso `n_jobs=-1`, que reparte los ajustes entre todos los núcleos de la computadora; el resultado es el mismo, solo cambia el tiempo.

In [6]:
grid = GridSearchCV(model, param_grid, scoring="roc_auc", cv=kf, n_jobs=-1)

%time grid.fit(X, y)
print(grid.best_params_)
print(grid.best_score_)

CPU times: total: 5h 16min
Wall time: 5h 25min 3s
{'mlpclassifier__hidden_layer_sizes': (6, 18, 8)}
0.9810595070560929


c:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [7]:
# Las 5 mejores redes de la rejilla
res_grid = pd.DataFrame(grid.cv_results_)
res_grid = res_grid[["param_mlpclassifier__hidden_layer_sizes", "mean_test_score", "std_test_score", "rank_test_score"]]
res_grid.columns = ["Red", "AUC promedio", "Desviación estándar", "Lugar"]
res_grid[res_grid["Lugar"] <= 5]

,Red,AUC promedio,Desviación estándar,Lugar
4,"(5, 14)",0.979133,0.015904,5
290,"(6, 13, 5)",0.980165,0.012714,2
373,"(6, 18, 8)",0.981060,0.012667,1
2287,"(6, 11, 7, 6)",0.979805,0.012447,4
3108,"(6, 19, 16, 5)",0.979824,0.011330,3


**6. Método 2: `RandomizedSearchCV`**

Uso la misma lista de redes, pero solo sorteo 100 (1.3 % de la rejilla), es decir, 1,000 entrenamientos.

In [8]:
rand = RandomizedSearchCV(model, param_grid, n_iter=100, scoring="roc_auc", cv=kf,
                          n_jobs=-1, random_state=42)

%time rand.fit(X, y)
print(rand.best_params_)
print(rand.best_score_)

CPU times: total: 12.6 s
Wall time: 2min 59s
{'mlpclassifier__hidden_layer_sizes': (6, 11, 7, 10)}
0.9742987422460265


c:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [9]:
# ¿En qué lugar de la rejilla completa quedó la red que encontró?
red_rand = rand.best_params_["mlpclassifier__hidden_layer_sizes"]
print("Lugar en la rejilla:", grid.cv_results_["rank_test_score"][capas.index(red_rand)])

Lugar en la rejilla: 156


**7. Método 3: Optuna**

En la función objetivo le pido a Optuna el número de capas y las neuronas de cada una, y solo pido las de la capa 3 o 4 si la red las tiene. Cada red la califico con el ciclo de K-Fold, con los mismos 10 grupos de `kf`, y regreso el AUC promedio. También le doy 100 intentos para compararlo en igualdad con `RandomizedSearchCV`.

In [10]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    n_capas = trial.suggest_int("n_capas", 2, 4)
    c1 = trial.suggest_int("c1", 5, 10)
    c2 = trial.suggest_int("c2", 10, 20)
    red = [c1, c2]
    if n_capas >= 3:
        red.append(trial.suggest_int("c3", 5, 20))
    if n_capas == 4:
        red.append(trial.suggest_int("c4", 5, 10))

    auc = []
    for train, test in kf.split(X):
        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", MLPClassifier(hidden_layer_sizes=tuple(red), max_iter=500, random_state=42))
        ])
        pipeline.fit(X.iloc[train], y.iloc[train])
        prob = pipeline.predict_proba(X.iloc[test])[:, 1]
        auc.append(roc_auc_score(y.iloc[test], prob))

    return np.mean(auc)

In [11]:
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))

%time study.optimize(objective, n_trials=100)
print(study.best_params)
print(study.best_value)

c:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the opti

CPU times: total: 7min 44s
Wall time: 7min 54s
{'n_capas': 2, 'c1': 5, 'c2': 14}
0.9791330802591782


c:\Users\carlo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [12]:
# Red que encontró Optuna, cuántas redes distintas probó y su lugar en la rejilla
p = study.best_params
red_opt = [p["c1"], p["c2"]]
if p["n_capas"] >= 3:
    red_opt.append(p["c3"])
if p["n_capas"] == 4:
    red_opt.append(p["c4"])
red_opt = tuple(red_opt)

intentos = study.trials_dataframe()
print("Red:", red_opt)
redes_probadas = intentos[["params_n_capas", "params_c1", "params_c2", "params_c3", "params_c4"]].drop_duplicates()
print("Redes distintas probadas:", redes_probadas.shape[0])
print("Lugar en la rejilla:", grid.cv_results_["rank_test_score"][capas.index(red_opt)])

Red: (5, 14)
Redes distintas probadas: 66
Lugar en la rejilla: 5


**8. Comparación de los tres métodos**

In [13]:
comparacion = pd.DataFrame([
    ["GridSearchCV", grid.best_params_["mlpclassifier__hidden_layer_sizes"], grid.best_score_, len(capas),
     grid.cv_results_["rank_test_score"][capas.index(grid.best_params_["mlpclassifier__hidden_layer_sizes"])]],
    ["RandomizedSearchCV", red_rand, rand.best_score_, 100,
     grid.cv_results_["rank_test_score"][capas.index(red_rand)]],
    ["Optuna", red_opt, study.best_value, 100,
     grid.cv_results_["rank_test_score"][capas.index(red_opt)]],
], columns=["Método", "Mejor red", "AUC promedio", "Redes evaluadas", "Lugar en la rejilla"])
comparacion

,Método,Mejor red,AUC promedio,Redes evaluadas,Lugar en la rejilla
0,GridSearchCV,"(6, 18, 8)",0.981060,7458,1
1,RandomizedSearchCV,"(6, 11, 7, 10)",0.974299,100,156
2,Optuna,"(5, 14)",0.979133,100,5


# Interpretación

Los tres métodos llegaron a redes con un AUC promedio muy parecido, pero con un costo muy distinto. GridSearchCV probó las 7,458 redes y encontró la mejor de toda la rejilla, la red (6, 18, 8) con un AUC de 0.981, aunque tardó 5 horas y 25 minutos en hacer los 74,580 entrenamientos, y eso que los repartió entre todos los núcleos de la computadora. RandomizedSearchCV sorteó 100 redes y en 3 minutos llegó a la (6, 11, 7, 10) con un AUC de 0.974, que quedó en el lugar 156 de la rejilla. Optuna también tuvo 100 intentos, pero como usa los resultados anteriores para decidir qué probar después se fue concentrando en una zona y repitió varias redes (solo probó 66 distintas), y en 8 minutos encontró la (5, 14) con un AUC de 0.979, que es la quinta mejor de las 7,458.

La diferencia entre el AUC de GridSearchCV y el de Optuna es de apenas 0.002, mucho menor que la desviación estándar del AUC entre los 10 grupos del K-Fold, que en las mejores redes anda cerca de 0.013, así que en la práctica las cinco mejores redes de la rejilla son igual de buenas y no se puede decir que la red de 3 capas sea mejor que la de 2 capas que encontró Optuna. Por eso revisar toda la rejilla no se justifica en este problema, ya que Optuna llegó prácticamente al mismo resultado en unas 40 veces menos tiempo aunque corrió en un solo núcleo.

RandomizedSearchCV fue el más rápido, pero su mejor red quedó más abajo porque solo revisó el 1.3% de la rejilla y cada sorteo es independiente de los anteriores. Además, como las redes de 4 capas son el 85% de la lista, 83 de sus 100 sorteos fueron redes de 4 capas, 17 fueron de 3 capas y ninguno de 2 capas, por lo que nunca probó redes como la (5, 14). Aun así, quedar en el lugar 156 de 7,458 lo pone dentro del 2% de las mejores redes, y lo hizo en 3 minutos.